In [ ]:
import pandas as pd
import numpy as np
import io

# Read the  file and repair missing newlines before the header
with open('arg_trial_data.csv', 'r') as f:
    raw_content = f.read()

# Replace instances where the header is glued directly to the previous line
header_str = 'participant_id,seniority'
repaired_content = raw_content.replace(header_str, '\n' + header_str)

# Load into pandas, automatically skipping any duplicate header rows that appear in the middle
df = pd.read_csv(io.StringIO(repaired_content))
df = df[df['participant_id'] != 'participant_id'].reset_index(drop=True)

# Convert types
df['correct'] = pd.to_numeric(df['correct'])
df['followed_ai_recommendation'] = pd.to_numeric(df['followed_ai_recommendation'])
df['trust_rating'] = pd.to_numeric(df['trust_rating'])
df['decision_time_ms'] = pd.to_numeric(df['decision_time_ms'])

# ── Trust calibration MAE ──────────────────────────────────────────
# Map trust rating (1-5 scale) to a 0-1 range to align with decision accuracy (0 or 1)
df['trust_norm'] = (df['trust_rating'] - 1) / 4
df['calib_error'] = abs(df['trust_norm'] - df['correct'])

# Participant-level mean per condition
calib = df.groupby(['participant_id', 'condition'])['calib_error'].mean().reset_index()
calib_stats = calib.groupby('condition')['calib_error'].agg(['mean', 'std'])
print("Trust calibration MAE:")
print(calib_stats.round(3))

# ── Decision time ──────────────────────────────────────────────────
time_stats = df.groupby('condition')['decision_time_ms'].agg(['mean', 'std'])
print("\nDecision time (ms):")
print(time_stats.round(1))

# ── Decision accuracy (proportion correct) ─────────────────────────
acc = df.groupby(['participant_id', 'condition'])['correct'].mean().reset_index()
acc_stats = acc.groupby('condition')['correct'].agg(['mean', 'std'])
print("\nDecision accuracy (proportion):")
print(acc_stats.round(3))

# ── Follow-AI rate (proportion) ────────────────────────────────────
fai = df.groupby(['participant_id', 'condition'])['followed_ai_recommendation'].mean().reset_index()
fai_stats = fai.groupby('condition')['followed_ai_recommendation'].agg(['mean', 'std'])
print("\nFollow-AI rate (proportion):")
print(fai_stats.round(3))


Trust calibration MAE:
                mean    std
condition                  
explained      0.331  0.063
non-explained  0.333  0.068

Decision time (ms):
                 mean     std
condition                    
explained      5218.9  8114.2
non-explained  4080.2  3031.5

Decision accuracy (proportion):
                mean    std
condition                  
explained      0.652  0.048
non-explained  0.662  0.050

Follow-AI rate (proportion):
               mean    std
condition                 
explained      0.98  0.055
non-explained  0.98  0.055
